In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv('../dataset/open-meteo_weather.csv')
df.head()

In [ ]:
df.set_index('time', inplace=True)

df.head()

In [ ]:
print(df.shape)

In [ ]:
df.info()

# Grafik

In [ ]:
# Pastikan index sudah dalam format datetime
df.index = pd.to_datetime(df.index)

# Resampling data per bulan dengan rata-rata
monthly_data = df.resample('M').mean()

# Setup plot
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(18, 15))
fig.tight_layout(pad=5.0)

# Nama kolom untuk plotting
columns = [
    'temperature_2m (°C)',
    'relative_humidity_2m (%)',
    'dew_point_2m (°C)',
    'apparent_temperature (°C)',
    'rain (mm)',
    'cloud_cover (%)',
    'cloud_cover_low (%)',
    'cloud_cover_mid (%)',
    'cloud_cover_high (%)'
]

# Loop untuk plotting setiap fitur
for i, column in enumerate(columns):
    ax = axes[i // 3, i % 3]  # Mengatur subplot berdasarkan posisi
    ax.plot(monthly_data.index, monthly_data[column], marker='o', linestyle='-')
    ax.set_title(column)
    ax.set_xlabel('Month')
    ax.set_ylabel('Average Value')

# Ajust space between plots
plt.subplots_adjust(hspace=0.4, wspace=0.4)

# Tampilkan plot
plt.show()

In [ ]:
# Pastikan index sudah dalam format datetime
df.index = pd.to_datetime(df.index)

# Resampling data per hari dengan rata-rata
daily_data = df.resample('D').mean()

# Membuat kolom grup untuk tiap 7 hari (dimulai dari hari pertama data)
daily_data['7day_group'] = ((daily_data.index - daily_data.index[0]).days // 7) + 1

# Nama kolom yang akan di-plot
columns = [
    'temperature_2m (°C)',
    'relative_humidity_2m (%)',
    'dew_point_2m (°C)',
    'apparent_temperature (°C)',
    'rain (mm)',
    'cloud_cover (%)',
    'cloud_cover_low (%)',
    'cloud_cover_mid (%)',
    'cloud_cover_high (%)'
]

# Mendapatkan grup 7 hari unik
unique_groups = sorted(daily_data['7day_group'].unique())
n_groups = len(unique_groups)

# Membuat subplot untuk tiap grup 7 hari
fig, axes = plt.subplots(nrows=n_groups, ncols=1, figsize=(18, 5 * n_groups))
if n_groups == 1:
    axes = [axes]

for ax, group in zip(axes, unique_groups):
    group_data = daily_data[daily_data['7day_group'] == group]
    # Plot tiap fitur dalam satu grafik untuk 7 hari tersebut
    for col in columns:
        ax.plot(group_data.index, group_data[col], marker='o', linestyle='-', label=col)
    # Menentukan rentang tanggal untuk judul
    start_date = group_data.index.min().strftime('%Y-%m-%d')
    end_date = group_data.index.max().strftime('%Y-%m-%d')
    ax.set_title(f"Data dari {start_date} hingga {end_date} (Grup {group})")
    ax.set_xlabel("Tanggal")
    ax.set_ylabel("Nilai")
    ax.legend()

plt.tight_layout()
plt.show()


# Correlation

In [ ]:
columns = [
    'temperature_2m (°C)',
    'relative_humidity_2m (%)',
    'dew_point_2m (°C)',
    'apparent_temperature (°C)',
    'rain (mm)',
    'cloud_cover (%)',
    'cloud_cover_low (%)',
    'cloud_cover_mid (%)',
    'cloud_cover_high (%)'
]

daily_data = df.resample('D').mean()
radiation_data = daily_data[columns]
corr_matrix = radiation_data.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Matrix Weather (W/m²)')
plt.show()

In [ ]:
from itertools import combinations
from scipy.stats import pearsonr

for col1, col2 in combinations(columns, 2):
    series1 = daily_data[col1].dropna()
    series2 = daily_data[col2].dropna()
    
    # Menyelaraskan indeks jika diperlukan
    common_index = series1.index.intersection(series2.index)
    series1 = series1.loc[common_index]
    series2 = series2.loc[common_index]
    
    corr_coef, p_value = pearsonr(series1, series2)
    print(f"Pearson correlation between '{col1}' and '{col2}': {corr_coef:.4f}, p-value: {p_value:.4f}")


# Uji statistik

In [ ]:
from statsmodels.tsa.stattools import adfuller 

for col in columns:
    series = daily_data[col].dropna()
    result = adfuller(series)
    adf_stat = result[0]
    p_value = result[1]
    crit_values = result[4]
    
    print(f'Uji ADF untuk kolom: {col}')
    print(f'ADF Statistic: {adf_stat:.4f}')
    print(f'p-value: {p_value:.4f}')
    print('Critical Values:')
    for key, value in crit_values.items():
        print(f'   {key}: {value:.4f}')
    
    if p_value < 0.05:
        print('=> Data stasioner (tolak H0)\n')
    else:
        print('=> Data tidak stasioner (gagal tolak H0)\n')

# Cek causation atau causality

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

# Tentukan jumlah lag maksimum yang ingin diuji (misal: 7 hari)
maxlag = 7

# Uji Granger Causality antar pasangan variabel
for col1 in columns:
    for col2 in columns:
        if col1 != col2:
            print(f'Uji Granger Causality: Apakah "{col1}" menyebabkan "{col2}"?')
            # Susun data dengan kolom target di depan, kemudian kolom penyebab
            data_for_test = daily_data[[col2, col1]].dropna()
            test_result = grangercausalitytests(data_for_test, maxlag=maxlag, verbose=False)
            
            # Simpan lag-lag yang signifikan
            significant_lags = []
            for lag in range(1, maxlag+1):
                p_value = test_result[lag][0]['ssr_chi2test'][1]
                print(f'Lag {lag}: p-value = {p_value:.4f}')
                if p_value < 0.05:
                    significant_lags.append(lag)
            
            # Menampilkan kesimpulan berdasarkan hasil uji
            if significant_lags:
                print(f"Kesimpulan: Ada bukti bahwa '{col1}' menyebabkan '{col2}' pada lag {significant_lags}.")
            else:
                print(f"Kesimpulan: Tidak ada bukti bahwa '{col1}' menyebabkan '{col2}'.")
            print("\n")
